# Polynomial Regression - From Scratch Implementation

## Table of Contents
1. [Theory & Mathematical Foundation](#theory)
2. [Implementation from Scratch](#implementation)
3. [Training & Optimization](#training)
4. [Diagnostics & Evaluation](#diagnostics)
5. [Visualizations](#visualizations)
6. [Use Cases & Guidelines](#use-cases)
7. [Comparison with sklearn](#comparison)

In [ ]:
# Import necessary libraries
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

## 1. Theory & Mathematical Foundation <a id='theory'></a>

### What is Polynomial Regression?

Polynomial regression is an extension of linear regression that models the relationship between the independent variable x and the dependent variable y as an nth degree polynomial. It allows us to capture **non-linear relationships** while still using the linear regression framework.

### Mathematical Formulation

#### Polynomial Model
$$y = \beta_0 + \beta_1 x + \beta_2 x^2 + \beta_3 x^3 + ... + \beta_n x^n + \epsilon$$

Where:
- $\beta_0, \beta_1, ..., \beta_n$ are the coefficients
- $n$ is the degree of the polynomial
- $\epsilon$ is the error term

#### Feature Transformation
The key insight is that polynomial regression is **linear in the coefficients**, even though it's non-linear in x. We transform the input:

$$x \rightarrow [1, x, x^2, x^3, ..., x^n]$$

Then apply standard linear regression to these transformed features.

#### Normal Equation (Closed-Form Solution)
$$\boldsymbol{\beta} = (\mathbf{X}^T \mathbf{X})^{-1} \mathbf{X}^T \mathbf{y}$$

Where $\mathbf{X}$ is the design matrix with polynomial features.

### Degree Selection: Overfitting vs Underfitting

| Degree | Effect | Problem |
|--------|--------|--------|
| Too Low | Model too simple | **Underfitting** (high bias) |
| Just Right | Captures true pattern | Good generalization |
| Too High | Fits noise | **Overfitting** (high variance) |

### Bias-Variance Tradeoff

$$\text{Total Error} = \text{Bias}^2 + \text{Variance} + \text{Irreducible Error}$$

- **Low degree (high bias)**: Model cannot capture the underlying pattern
- **High degree (high variance)**: Model is too sensitive to training data
- **Optimal degree**: Balances bias and variance for best generalization

### Time Complexity
- Feature generation: O(n_samples * degree)
- Training (Normal Equation): O(n_features^3) where n_features = degree + 1
- Prediction: O(n_samples * n_features)

## 2. Implementation from Scratch <a id='implementation'></a>

In [ ]:
class PolynomialRegression:
    """
    Polynomial Regression implementation from scratch using NumPy.
    
    This class generates polynomial features from input data and fits
    a linear regression model using the normal equation.
    
    Parameters:
    -----------
    degree : int, default=2
        The degree of the polynomial features.
    include_bias : bool, default=True
        Whether to include a bias (intercept) term.
    regularization : float, default=0.0
        L2 regularization strength (Ridge). Set to 0 for no regularization.
    
    Attributes:
    -----------
    coefficients : ndarray
        Fitted coefficients after training.
    """
    
    def __init__(self, degree=2, include_bias=True, regularization=0.0):
        self.degree = degree
        self.include_bias = include_bias
        self.regularization = regularization
        self.coefficients = None
        self._feature_names = None
    
    def _generate_polynomial_features(self, X):
        """
        Generate polynomial features from input data.
        
        For a single feature x and degree=3, generates:
        [1, x, x^2, x^3] if include_bias=True
        [x, x^2, x^3] if include_bias=False
        
        Parameters:
        -----------
        X : ndarray, shape (n_samples,) or (n_samples, 1)
            Input features.
        
        Returns:
        --------
        X_poly : ndarray, shape (n_samples, n_features)
            Polynomial feature matrix.
        """
        # Ensure X is 1D
        X = np.asarray(X).flatten()
        n_samples = len(X)
        
        # Determine starting power based on include_bias
        start_power = 0 if self.include_bias else 1
        
        # Generate polynomial features: [1, x, x^2, ..., x^degree]
        # Using column_stack for efficient memory layout
        features = [np.power(X, p) for p in range(start_power, self.degree + 1)]
        X_poly = np.column_stack(features)
        
        # Store feature names for interpretability
        self._feature_names = [f'x^{p}' for p in range(start_power, self.degree + 1)]
        if self.include_bias:
            self._feature_names[0] = '1 (bias)'
        
        return X_poly
    
    def fit(self, X, y):
        """
        Fit the polynomial regression model.
        
        Uses the normal equation with optional L2 regularization:
        beta = (X^T X + lambda*I)^{-1} X^T y
        
        Parameters:
        -----------
        X : array-like, shape (n_samples,) or (n_samples, 1)
            Training data.
        y : array-like, shape (n_samples,)
            Target values.
        
        Returns:
        --------
        self : object
            Returns self for method chaining.
        """
        # Convert to numpy arrays
        X = np.asarray(X)
        y = np.asarray(y).flatten()
        
        # Generate polynomial features
        X_poly = self._generate_polynomial_features(X)
        
        # Compute normal equation: (X^T X + lambda*I)^{-1} X^T y
        n_features = X_poly.shape[1]
        
        # X^T X
        XtX = np.dot(X_poly.T, X_poly)
        
        # Add regularization (don't regularize bias term)
        if self.regularization > 0:
            reg_matrix = self.regularization * np.eye(n_features)
            if self.include_bias:
                reg_matrix[0, 0] = 0  # Don't regularize bias
            XtX += reg_matrix
        
        # X^T y
        Xty = np.dot(X_poly.T, y)
        
        # Solve using more numerically stable method
        try:
            # Try Cholesky decomposition (faster for positive definite matrices)
            L = np.linalg.cholesky(XtX)
            z = np.linalg.solve(L, Xty)
            self.coefficients = np.linalg.solve(L.T, z)
        except np.linalg.LinAlgError:
            # Fall back to pseudo-inverse for ill-conditioned matrices
            self.coefficients = np.linalg.lstsq(XtX, Xty, rcond=None)[0]
        
        return self
    
    def predict(self, X):
        """
        Predict using the polynomial regression model.
        
        Parameters:
        -----------
        X : array-like, shape (n_samples,) or (n_samples, 1)
            Samples to predict.
        
        Returns:
        --------
        y_pred : ndarray, shape (n_samples,)
            Predicted values.
        """
        if self.coefficients is None:
            raise ValueError("Model has not been fitted. Call fit() first.")
        
        X_poly = self._generate_polynomial_features(X)
        return np.dot(X_poly, self.coefficients)
    
    def score(self, X, y):
        """
        Calculate R^2 score (coefficient of determination).
        
        Parameters:
        -----------
        X : array-like
            Test samples.
        y : array-like
            True values.
        
        Returns:
        --------
        score : float
            R^2 score.
        """
        y = np.asarray(y).flatten()
        y_pred = self.predict(X)
        
        # Total sum of squares
        ss_tot = np.sum((y - np.mean(y)) ** 2)
        
        # Residual sum of squares
        ss_res = np.sum((y - y_pred) ** 2)
        
        # R^2 = 1 - SS_res / SS_tot
        return 1 - (ss_res / ss_tot) if ss_tot > 0 else 0.0
    
    def get_equation(self):
        """
        Return the polynomial equation as a string.
        
        Returns:
        --------
        equation : str
            String representation of the fitted polynomial.
        """
        if self.coefficients is None:
            return "Model not fitted"
        
        terms = []
        start_idx = 0 if self.include_bias else 1
        
        for i, coef in enumerate(self.coefficients):
            power = i if self.include_bias else i + 1
            if power == 0:
                terms.append(f"{coef:.4f}")
            elif power == 1:
                terms.append(f"{coef:+.4f}x")
            else:
                terms.append(f"{coef:+.4f}x^{power}")
        
        return "y = " + " ".join(terms)

In [ ]:
# Quick test of the implementation
print("Testing PolynomialRegression class...")
print("="*50)

# Create simple test data: y = 2 + 3x + x^2
X_test = np.array([1, 2, 3, 4, 5])
y_test = 2 + 3*X_test + X_test**2  # True relationship

# Fit model
model = PolynomialRegression(degree=2)
model.fit(X_test, y_test)

print(f"True equation: y = 2 + 3x + x^2")
print(f"Fitted equation: {model.get_equation()}")
print(f"\nCoefficients: {model.coefficients}")
print(f"R^2 score: {model.score(X_test, y_test):.6f}")

## 3. Training & Optimization <a id='training'></a>

Let's generate synthetic polynomial data with noise and train our model.

In [ ]:
def generate_polynomial_data(n_samples=100, true_degree=3, noise_std=0.5, 
                              x_range=(-3, 3), random_state=42):
    """
    Generate synthetic polynomial data with noise.
    
    Parameters:
    -----------
    n_samples : int
        Number of samples to generate.
    true_degree : int
        Degree of the true polynomial.
    noise_std : float
        Standard deviation of Gaussian noise.
    x_range : tuple
        Range of x values (min, max).
    random_state : int
        Random seed for reproducibility.
    
    Returns:
    --------
    X : ndarray
        Input features.
    y : ndarray
        Target values (with noise).
    y_true : ndarray
        True target values (without noise).
    true_coefficients : ndarray
        True polynomial coefficients.
    """
    np.random.seed(random_state)
    
    # Generate x values
    X = np.linspace(x_range[0], x_range[1], n_samples)
    
    # Generate random coefficients for the true polynomial
    # Using smaller coefficients for higher powers to avoid exploding values
    true_coefficients = np.random.randn(true_degree + 1)
    true_coefficients = true_coefficients / (np.arange(true_degree + 1) + 1)
    
    # Compute true y values
    y_true = np.zeros_like(X)
    for i, coef in enumerate(true_coefficients):
        y_true += coef * np.power(X, i)
    
    # Add noise
    noise = np.random.normal(0, noise_std, n_samples)
    y = y_true + noise
    
    return X, y, y_true, true_coefficients


# Generate synthetic data
print("Generating synthetic polynomial data...")
print("="*50)

X, y, y_true, true_coeffs = generate_polynomial_data(
    n_samples=100, 
    true_degree=3, 
    noise_std=0.3,
    x_range=(-2, 2)
)

# Display true polynomial
true_eq_terms = [f"{c:.3f}x^{i}" for i, c in enumerate(true_coeffs)]
print(f"True polynomial: y = {' + '.join(true_eq_terms)}")
print(f"\nData shape: X={X.shape}, y={y.shape}")
print(f"X range: [{X.min():.2f}, {X.max():.2f}]")
print(f"y range: [{y.min():.2f}, {y.max():.2f}]")

In [ ]:
# Split data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Sort for proper visualization later
train_idx = np.argsort(X_train)
X_train_sorted = X_train[train_idx]
y_train_sorted = y_train[train_idx]

test_idx = np.argsort(X_test)
X_test_sorted = X_test[test_idx]
y_test_sorted = y_test[test_idx]

print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")

In [ ]:
# Train models with different degrees
degrees = [1, 2, 3, 4, 5, 7, 10, 15]
models = {}

print("Training models with different polynomial degrees...")
print("="*70)
print(f"{'Degree':<8} {'Train MSE':<12} {'Test MSE':<12} {'Train R2':<12} {'Test R2':<12}")
print("-"*70)

for degree in degrees:
    # Create and train model
    model = PolynomialRegression(degree=degree)
    model.fit(X_train, y_train)
    models[degree] = model
    
    # Calculate metrics
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    train_mse = np.mean((y_train - y_train_pred) ** 2)
    test_mse = np.mean((y_test - y_test_pred) ** 2)
    train_r2 = model.score(X_train, y_train)
    test_r2 = model.score(X_test, y_test)
    
    print(f"{degree:<8} {train_mse:<12.4f} {test_mse:<12.4f} {train_r2:<12.4f} {test_r2:<12.4f}")

## 4. Diagnostics & Evaluation <a id='diagnostics'></a>

In [ ]:
def plot_mse_vs_degree(X_train, y_train, X_test, y_test, max_degree=15):
    """
    Plot MSE vs polynomial degree to visualize overfitting.
    
    Parameters:
    -----------
    X_train, y_train : Training data
    X_test, y_test : Test data
    max_degree : Maximum polynomial degree to test
    
    Returns:
    --------
    results : dict
        Dictionary with train/test MSE for each degree.
    """
    degrees = range(1, max_degree + 1)
    train_mse = []
    test_mse = []
    
    for degree in degrees:
        model = PolynomialRegression(degree=degree)
        model.fit(X_train, y_train)
        
        y_train_pred = model.predict(X_train)
        y_test_pred = model.predict(X_test)
        
        train_mse.append(np.mean((y_train - y_train_pred) ** 2))
        test_mse.append(np.mean((y_test - y_test_pred) ** 2))
    
    # Plot
    fig, ax = plt.subplots(figsize=(10, 6))
    
    ax.plot(degrees, train_mse, 'b-o', label='Training MSE', linewidth=2, markersize=6)
    ax.plot(degrees, test_mse, 'r-s', label='Test MSE', linewidth=2, markersize=6)
    
    # Find optimal degree (lowest test MSE)
    optimal_degree = degrees[np.argmin(test_mse)]
    ax.axvline(x=optimal_degree, color='green', linestyle='--', 
               label=f'Optimal Degree = {optimal_degree}', alpha=0.7)
    
    # Add regions
    ax.axvspan(1, optimal_degree - 0.5, alpha=0.1, color='blue', label='Underfitting Region')
    ax.axvspan(optimal_degree + 0.5, max_degree, alpha=0.1, color='red', label='Overfitting Region')
    
    ax.set_xlabel('Polynomial Degree', fontsize=12)
    ax.set_ylabel('Mean Squared Error', fontsize=12)
    ax.set_title('MSE vs Polynomial Degree: Detecting Overfitting', fontsize=14)
    ax.legend(loc='upper right')
    ax.set_xticks(degrees)
    ax.grid(True, alpha=0.3)
    
    # Use log scale if values vary widely
    if max(test_mse) / min(test_mse) > 100:
        ax.set_yscale('log')
    
    plt.tight_layout()
    plt.show()
    
    return {'degrees': list(degrees), 'train_mse': train_mse, 'test_mse': test_mse,
            'optimal_degree': optimal_degree}


# Plot MSE vs degree
results = plot_mse_vs_degree(X_train, y_train, X_test, y_test, max_degree=15)

In [ ]:
def cross_validation_degree_selection(X, y, max_degree=10, cv=5):
    """
    Use cross-validation to select the optimal polynomial degree.
    
    Parameters:
    -----------
    X : array-like
        Input features.
    y : array-like
        Target values.
    max_degree : int
        Maximum degree to consider.
    cv : int
        Number of cross-validation folds.
    
    Returns:
    --------
    results : dict
        Cross-validation results for each degree.
    """
    from sklearn.model_selection import KFold
    
    X = np.asarray(X)
    y = np.asarray(y)
    
    degrees = range(1, max_degree + 1)
    cv_scores_mean = []
    cv_scores_std = []
    
    kf = KFold(n_splits=cv, shuffle=True, random_state=42)
    
    for degree in degrees:
        fold_scores = []
        
        for train_idx, val_idx in kf.split(X):
            X_fold_train, X_fold_val = X[train_idx], X[val_idx]
            y_fold_train, y_fold_val = y[train_idx], y[val_idx]
            
            model = PolynomialRegression(degree=degree)
            model.fit(X_fold_train, y_fold_train)
            
            # Negative MSE (sklearn convention: higher is better)
            y_pred = model.predict(X_fold_val)
            mse = np.mean((y_fold_val - y_pred) ** 2)
            fold_scores.append(-mse)
        
        cv_scores_mean.append(np.mean(fold_scores))
        cv_scores_std.append(np.std(fold_scores))
    
    cv_scores_mean = np.array(cv_scores_mean)
    cv_scores_std = np.array(cv_scores_std)
    
    # Plot
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Plot mean with error bars (negative MSE, so higher is better)
    ax.errorbar(degrees, -cv_scores_mean, yerr=cv_scores_std, 
                fmt='o-', capsize=5, capthick=2, linewidth=2, 
                markersize=8, label='CV Mean MSE (+/- 1 std)')
    
    # Highlight optimal degree
    optimal_idx = np.argmax(cv_scores_mean)  # highest (least negative) score
    optimal_degree = list(degrees)[optimal_idx]
    
    ax.scatter([optimal_degree], [-cv_scores_mean[optimal_idx]], 
               s=200, c='red', marker='*', zorder=5,
               label=f'Optimal Degree = {optimal_degree}')
    
    ax.set_xlabel('Polynomial Degree', fontsize=12)
    ax.set_ylabel('Mean Squared Error (CV)', fontsize=12)
    ax.set_title(f'{cv}-Fold Cross-Validation for Degree Selection', fontsize=14)
    ax.legend(loc='upper right')
    ax.set_xticks(list(degrees))
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nOptimal degree by cross-validation: {optimal_degree}")
    print(f"CV MSE at optimal degree: {-cv_scores_mean[optimal_idx]:.4f} (+/- {cv_scores_std[optimal_idx]:.4f})")
    
    return {
        'degrees': list(degrees),
        'cv_mse_mean': list(-cv_scores_mean),
        'cv_mse_std': list(cv_scores_std),
        'optimal_degree': optimal_degree
    }


# Perform cross-validation
cv_results = cross_validation_degree_selection(X, y, max_degree=12, cv=5)

In [ ]:
# Residual analysis for the optimal model
optimal_degree = cv_results['optimal_degree']
optimal_model = PolynomialRegression(degree=optimal_degree)
optimal_model.fit(X_train, y_train)

# Predictions
y_train_pred = optimal_model.predict(X_train)
y_test_pred = optimal_model.predict(X_test)

# Residuals
train_residuals = y_train - y_train_pred
test_residuals = y_test - y_test_pred

# Plot residual diagnostics
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# 1. Residuals vs Fitted
axes[0, 0].scatter(y_train_pred, train_residuals, alpha=0.6, label='Train')
axes[0, 0].scatter(y_test_pred, test_residuals, alpha=0.6, label='Test')
axes[0, 0].axhline(y=0, color='r', linestyle='--', linewidth=2)
axes[0, 0].set_xlabel('Fitted Values')
axes[0, 0].set_ylabel('Residuals')
axes[0, 0].set_title('Residuals vs Fitted Values')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 2. Histogram of residuals
axes[0, 1].hist(train_residuals, bins=20, alpha=0.6, label='Train', density=True)
axes[0, 1].hist(test_residuals, bins=15, alpha=0.6, label='Test', density=True)
axes[0, 1].set_xlabel('Residuals')
axes[0, 1].set_ylabel('Density')
axes[0, 1].set_title('Distribution of Residuals')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# 3. Q-Q Plot
from scipy import stats
stats.probplot(test_residuals, dist="norm", plot=axes[1, 0])
axes[1, 0].set_title('Q-Q Plot (Test Residuals)')
axes[1, 0].grid(True, alpha=0.3)

# 4. Actual vs Predicted
axes[1, 1].scatter(y_test, y_test_pred, alpha=0.6)
min_val = min(y_test.min(), y_test_pred.min())
max_val = max(y_test.max(), y_test_pred.max())
axes[1, 1].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')
axes[1, 1].set_xlabel('Actual Values')
axes[1, 1].set_ylabel('Predicted Values')
axes[1, 1].set_title('Actual vs Predicted (Test Set)')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle(f'Residual Diagnostics (Degree = {optimal_degree})', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Print metrics
print(f"\nModel Performance (Degree = {optimal_degree}):")
print(f"  Train MSE: {np.mean(train_residuals**2):.4f}")
print(f"  Test MSE: {np.mean(test_residuals**2):.4f}")
print(f"  Train R2: {optimal_model.score(X_train, y_train):.4f}")
print(f"  Test R2: {optimal_model.score(X_test, y_test):.4f}")

## 5. Visualizations <a id='visualizations'></a>

In [ ]:
def plot_fit_curves(X, y, degrees_to_plot, X_train=None, y_train=None):
    """
    Plot polynomial fit curves for different degrees.
    
    Parameters:
    -----------
    X, y : Full dataset for visualization
    degrees_to_plot : list of degrees to visualize
    X_train, y_train : Training data used for fitting
    """
    if X_train is None:
        X_train, y_train = X, y
    
    # Create dense x values for smooth curves
    X_plot = np.linspace(X.min() - 0.5, X.max() + 0.5, 200)
    
    n_plots = len(degrees_to_plot)
    n_cols = min(3, n_plots)
    n_rows = (n_plots + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
    if n_plots == 1:
        axes = np.array([axes])
    axes = axes.flatten()
    
    for idx, degree in enumerate(degrees_to_plot):
        ax = axes[idx]
        
        # Fit model
        model = PolynomialRegression(degree=degree)
        model.fit(X_train, y_train)
        
        # Predict
        y_plot = model.predict(X_plot)
        
        # Plot
        ax.scatter(X, y, alpha=0.5, s=30, label='Data', color='blue')
        ax.plot(X_plot, y_plot, 'r-', linewidth=2, label=f'Degree {degree} fit')
        
        # Calculate MSE
        y_pred = model.predict(X)
        mse = np.mean((y - y_pred) ** 2)
        r2 = model.score(X, y)
        
        ax.set_xlabel('x')
        ax.set_ylabel('y')
        ax.set_title(f'Degree = {degree}\nMSE = {mse:.3f}, R2 = {r2:.3f}')
        ax.legend(loc='upper left')
        ax.grid(True, alpha=0.3)
    
    # Hide empty subplots
    for idx in range(n_plots, len(axes)):
        axes[idx].set_visible(False)
    
    plt.suptitle('Polynomial Regression Fits for Different Degrees', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()


# Plot fit curves for various degrees
plot_fit_curves(X, y, degrees_to_plot=[1, 2, 3, 5, 10, 15], X_train=X_train, y_train=y_train)

In [ ]:
def demonstrate_overfitting(n_samples=30, true_degree=3, noise_std=0.5):
    """
    Comprehensive demonstration of underfitting, good fit, and overfitting.
    
    Uses a smaller dataset to make overfitting more pronounced.
    """
    # Generate data with smaller sample size
    X_demo, y_demo, y_true_demo, _ = generate_polynomial_data(
        n_samples=n_samples,
        true_degree=true_degree,
        noise_std=noise_std,
        x_range=(-2, 2),
        random_state=123
    )
    
    # Dense x for plotting
    X_plot = np.linspace(-2.5, 2.5, 300)
    
    # Different scenarios
    scenarios = [
        (1, 'Underfitting (Degree 1)', 'High Bias'),
        (3, 'Good Fit (Degree 3)', 'Balanced'),
        (15, 'Overfitting (Degree 15)', 'High Variance')
    ]
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    colors = ['#e74c3c', '#27ae60', '#9b59b6']
    
    for idx, (degree, title, label) in enumerate(scenarios):
        ax = axes[idx]
        
        # Fit model
        model = PolynomialRegression(degree=degree)
        model.fit(X_demo, y_demo)
        
        # Predict
        y_plot = model.predict(X_plot)
        
        # Clip extreme values for visualization
        y_range = y_demo.max() - y_demo.min()
        y_plot = np.clip(y_plot, y_demo.min() - y_range, y_demo.max() + y_range)
        
        # Plot
        ax.scatter(X_demo, y_demo, s=50, alpha=0.7, color='blue', label='Training Data', zorder=3)
        ax.plot(X_plot, y_plot, linewidth=2.5, color=colors[idx], label=f'Model (degree={degree})', zorder=2)
        
        # Calculate metrics
        y_pred = model.predict(X_demo)
        mse = np.mean((y_demo - y_pred) ** 2)
        r2 = model.score(X_demo, y_demo)
        
        ax.set_xlabel('x', fontsize=11)
        ax.set_ylabel('y', fontsize=11)
        ax.set_title(f'{title}\n{label}\nTrain MSE={mse:.3f}, R2={r2:.3f}', fontsize=12)
        ax.legend(loc='upper left', fontsize=9)
        ax.grid(True, alpha=0.3)
        ax.set_ylim(y_demo.min() - y_range*0.3, y_demo.max() + y_range*0.3)
    
    plt.suptitle('Bias-Variance Tradeoff Demonstration', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()


# Demonstrate overfitting
demonstrate_overfitting(n_samples=25, true_degree=3, noise_std=0.5)

In [ ]:
def plot_bias_variance_tradeoff(X, y, max_degree=15, n_bootstraps=50):
    """
    Visualize bias-variance tradeoff using bootstrap sampling.
    
    Parameters:
    -----------
    X, y : Dataset
    max_degree : Maximum polynomial degree
    n_bootstraps : Number of bootstrap samples
    """
    degrees = range(1, max_degree + 1)
    n_samples = len(X)
    
    # Create test points
    X_test_points = np.linspace(X.min(), X.max(), 20)
    
    bias_squared = []
    variance = []
    mse_total = []
    
    for degree in degrees:
        # Store predictions from each bootstrap
        all_predictions = []
        
        for _ in range(n_bootstraps):
            # Bootstrap sample
            idx = np.random.choice(n_samples, size=n_samples, replace=True)
            X_boot, y_boot = X[idx], y[idx]
            
            # Fit model
            model = PolynomialRegression(degree=degree)
            model.fit(X_boot, y_boot)
            
            # Predict on test points
            y_pred = model.predict(X_test_points)
            all_predictions.append(y_pred)
        
        all_predictions = np.array(all_predictions)
        
        # Mean prediction across bootstraps
        mean_prediction = np.mean(all_predictions, axis=0)
        
        # Variance of predictions
        var = np.mean(np.var(all_predictions, axis=0))
        variance.append(var)
        
        # Bias (need true values - approximate with full dataset fit)
        full_model = PolynomialRegression(degree=degree)
        full_model.fit(X, y)
        y_true_approx = full_model.predict(X_test_points)
        bias_sq = np.mean((mean_prediction - y_true_approx) ** 2)
        bias_squared.append(bias_sq)
        
        # Total MSE
        mse = np.mean((y - full_model.predict(X)) ** 2)
        mse_total.append(mse)
    
    # Plot
    fig, ax = plt.subplots(figsize=(10, 6))
    
    ax.plot(degrees, variance, 'b-o', label='Variance', linewidth=2, markersize=6)
    ax.plot(degrees, bias_squared, 'r-s', label='Bias^2 (approx)', linewidth=2, markersize=6)
    ax.plot(degrees, mse_total, 'g-^', label='Total MSE', linewidth=2, markersize=6)
    
    ax.set_xlabel('Polynomial Degree', fontsize=12)
    ax.set_ylabel('Error', fontsize=12)
    ax.set_title('Bias-Variance Tradeoff', fontsize=14)
    ax.legend(loc='upper right')
    ax.set_xticks(list(degrees))
    ax.grid(True, alpha=0.3)
    
    # Annotations
    ax.annotate('High Bias\nLow Variance', xy=(2, max(variance)*0.7), fontsize=10, ha='center')
    ax.annotate('Low Bias\nHigh Variance', xy=(max_degree-2, max(variance)*0.7), fontsize=10, ha='center')
    
    plt.tight_layout()
    plt.show()


# Plot bias-variance tradeoff
plot_bias_variance_tradeoff(X, y, max_degree=12, n_bootstraps=30)

## 6. Use Cases & Guidelines <a id='use-cases'></a>

### When to Use Polynomial Regression

#### Good Use Cases:

1. **Known Polynomial Relationships**
   - Physics: projectile motion (quadratic), spring mechanics
   - Economics: cost curves, production functions
   - Biology: growth curves, dose-response relationships

2. **Curved Relationships in Data**
   - When scatter plot shows clear curvature
   - Residuals from linear regression show systematic patterns
   - Domain knowledge suggests non-linear relationship

3. **Low-Dimensional Problems**
   - Single feature or few features
   - Sufficient data relative to model complexity
   - Interpretability is important

4. **Baseline for Non-Linear Modeling**
   - Quick first attempt at capturing non-linearity
   - Benchmark before trying more complex models

#### When NOT to Use:

1. **High-Dimensional Data**
   - Number of polynomial features explodes: (n+d)! / (n! * d!)
   - For 10 features with degree 3: 286 polynomial features!
   - Use: Neural Networks, Gradient Boosting, Random Forest

2. **Unknown Functional Form**
   - When relationship might not be polynomial
   - Complex interactions between features
   - Use: Splines, GAMs, Tree-based methods

3. **Extrapolation Required**
   - High-degree polynomials behave erratically outside training range
   - Predictions can explode to infinity
   - Use: Constrained models, domain-specific models

4. **Large Datasets**
   - Matrix inversion becomes expensive
   - Feature explosion with high degree
   - Use: Neural Networks, Online Learning algorithms

### Degree Selection Strategies

| Strategy | Description | When to Use |
|----------|-------------|-------------|
| Cross-Validation | Select degree with lowest CV error | General purpose, most reliable |
| Information Criteria (AIC/BIC) | Balance fit vs complexity | When computational cost is concern |
| Validation Set | Hold out data for evaluation | Large datasets |
| Domain Knowledge | Use known physical relationships | Strong theoretical basis |
| Visual Inspection | Plot fits for different degrees | Quick exploratory analysis |

### Regularization Need

**Why Regularization is Important:**
- High-degree polynomials have many parameters
- Without regularization, coefficients can become very large
- Leads to wild oscillations and poor generalization

**Types of Regularization:**
- **Ridge (L2)**: Shrinks coefficients, keeps all features
- **Lasso (L1)**: Can zero out coefficients, feature selection
- **Elastic Net**: Combination of L1 and L2

**Rule of Thumb:**
- Degree > 3: Consider regularization
- Degree > 5: Regularization strongly recommended
- High degree with small dataset: Regularization essential

### Common Pitfalls

1. **Overfitting to Noise**
   - Problem: Model fits noise perfectly but fails on new data
   - Solution: Use cross-validation, regularization

2. **Numerical Instability**
   - Problem: X^n becomes very large/small
   - Solution: Feature scaling before polynomial expansion

3. **Multicollinearity**
   - Problem: x, x^2, x^3 are highly correlated
   - Solution: Use orthogonal polynomials or regularization

4. **Extrapolation Disaster**
   - Problem: Predictions explode outside training range
   - Solution: Limit predictions to training range, use appropriate models

In [ ]:
# Demonstrate regularization effect
def demonstrate_regularization(X, y, degree=10):
    """
    Show the effect of regularization on high-degree polynomial.
    """
    X_plot = np.linspace(X.min() - 0.5, X.max() + 0.5, 200)
    
    # Different regularization strengths
    reg_values = [0, 0.01, 0.1, 1.0]
    
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    
    for idx, reg in enumerate(reg_values):
        ax = axes[idx]
        
        # Fit model with regularization
        model = PolynomialRegression(degree=degree, regularization=reg)
        model.fit(X, y)
        
        # Predict
        y_plot = model.predict(X_plot)
        y_pred = model.predict(X)
        
        # Clip for visualization
        y_range = y.max() - y.min()
        y_plot = np.clip(y_plot, y.min() - y_range, y.max() + y_range)
        
        # Plot
        ax.scatter(X, y, alpha=0.5, s=30, label='Data')
        ax.plot(X_plot, y_plot, 'r-', linewidth=2, label=f'Fit')
        
        mse = np.mean((y - y_pred) ** 2)
        coef_norm = np.linalg.norm(model.coefficients)
        
        ax.set_xlabel('x')
        ax.set_ylabel('y')
        reg_str = 'None' if reg == 0 else f'{reg}'
        ax.set_title(f'Regularization: {reg_str}\nMSE={mse:.3f}, ||w||={coef_norm:.1f}')
        ax.legend(loc='upper left', fontsize=9)
        ax.grid(True, alpha=0.3)
        ax.set_ylim(y.min() - y_range*0.3, y.max() + y_range*0.3)
    
    plt.suptitle(f'Effect of L2 Regularization on Degree-{degree} Polynomial', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()


# Demonstrate regularization
demonstrate_regularization(X, y, degree=10)

## 7. Comparison with sklearn <a id='comparison'></a>

In [ ]:
# Compare our implementation with sklearn
print("Comparing our implementation with sklearn...")
print("="*70)

# Test parameters
test_degree = 4

# Our implementation
our_model = PolynomialRegression(degree=test_degree)
our_model.fit(X_train, y_train)
our_pred_train = our_model.predict(X_train)
our_pred_test = our_model.predict(X_test)

# sklearn implementation using Pipeline
sklearn_pipeline = Pipeline([
    ('poly_features', PolynomialFeatures(degree=test_degree, include_bias=True)),
    ('linear_reg', LinearRegression(fit_intercept=False))  # bias already in poly features
])
sklearn_pipeline.fit(X_train.reshape(-1, 1), y_train)
sklearn_pred_train = sklearn_pipeline.predict(X_train.reshape(-1, 1))
sklearn_pred_test = sklearn_pipeline.predict(X_test.reshape(-1, 1))

# Metrics comparison
print(f"\nPolynomial Degree: {test_degree}")
print("-"*70)
print(f"{'Metric':<25} {'Our Implementation':<20} {'sklearn':<20}")
print("-"*70)

our_train_mse = mean_squared_error(y_train, our_pred_train)
sklearn_train_mse = mean_squared_error(y_train, sklearn_pred_train)
print(f"{'Train MSE':<25} {our_train_mse:<20.6f} {sklearn_train_mse:<20.6f}")

our_test_mse = mean_squared_error(y_test, our_pred_test)
sklearn_test_mse = mean_squared_error(y_test, sklearn_pred_test)
print(f"{'Test MSE':<25} {our_test_mse:<20.6f} {sklearn_test_mse:<20.6f}")

our_train_r2 = our_model.score(X_train, y_train)
sklearn_train_r2 = r2_score(y_train, sklearn_pred_train)
print(f"{'Train R2':<25} {our_train_r2:<20.6f} {sklearn_train_r2:<20.6f}")

our_test_r2 = our_model.score(X_test, y_test)
sklearn_test_r2 = r2_score(y_test, sklearn_pred_test)
print(f"{'Test R2':<25} {our_test_r2:<20.6f} {sklearn_test_r2:<20.6f}")

# Prediction agreement
pred_diff = np.abs(our_pred_test - sklearn_pred_test)
print(f"\nPrediction Agreement:")
print(f"  Mean Absolute Difference: {np.mean(pred_diff):.8f}")
print(f"  Max Absolute Difference: {np.max(pred_diff):.8f}")

In [ ]:
# Compare coefficients
print("\nCoefficient Comparison:")
print("="*70)

sklearn_coefs = sklearn_pipeline.named_steps['linear_reg'].coef_
our_coefs = our_model.coefficients

print(f"{'Power':<10} {'Our Coef':<20} {'sklearn Coef':<20} {'Difference':<20}")
print("-"*70)

for i in range(len(our_coefs)):
    diff = abs(our_coefs[i] - sklearn_coefs[i])
    print(f"x^{i:<7} {our_coefs[i]:<20.8f} {sklearn_coefs[i]:<20.8f} {diff:<20.2e}")

In [ ]:
# Visual comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

X_plot = np.linspace(X.min() - 0.5, X.max() + 0.5, 200)

# Plot 1: Both implementations overlaid
ax1 = axes[0]
ax1.scatter(X, y, alpha=0.5, s=30, label='Data', color='blue')
ax1.plot(X_plot, our_model.predict(X_plot), 'r-', linewidth=2, label='Our Implementation')
ax1.plot(X_plot, sklearn_pipeline.predict(X_plot.reshape(-1, 1)), 'g--', 
         linewidth=2, label='sklearn', alpha=0.8)
ax1.set_xlabel('x')
ax1.set_ylabel('y')
ax1.set_title(f'Polynomial Regression (Degree={test_degree})\nOur vs sklearn')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Prediction comparison
ax2 = axes[1]
ax2.scatter(sklearn_pred_test, our_pred_test, alpha=0.7)
min_val = min(our_pred_test.min(), sklearn_pred_test.min())
max_val = max(our_pred_test.max(), sklearn_pred_test.max())
ax2.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Agreement')
ax2.set_xlabel('sklearn Predictions')
ax2.set_ylabel('Our Predictions')
ax2.set_title('Prediction Comparison on Test Set')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Compare with regularized sklearn (Ridge)
print("\nComparison with Regularized sklearn (Ridge):")
print("="*70)

test_degree = 8
reg_strength = 1.0

# Our implementation with regularization
our_ridge = PolynomialRegression(degree=test_degree, regularization=reg_strength)
our_ridge.fit(X_train, y_train)

# sklearn Ridge
sklearn_ridge_pipeline = Pipeline([
    ('poly_features', PolynomialFeatures(degree=test_degree, include_bias=True)),
    ('ridge', Ridge(alpha=reg_strength, fit_intercept=False))
])
sklearn_ridge_pipeline.fit(X_train.reshape(-1, 1), y_train)

print(f"Degree: {test_degree}, Regularization: {reg_strength}")
print("-"*70)

our_test_mse = mean_squared_error(y_test, our_ridge.predict(X_test))
sklearn_test_mse = mean_squared_error(y_test, sklearn_ridge_pipeline.predict(X_test.reshape(-1, 1)))

print(f"Our Ridge Test MSE: {our_test_mse:.6f}")
print(f"sklearn Ridge Test MSE: {sklearn_test_mse:.6f}")
print(f"\nCoefficient Norm (Our): {np.linalg.norm(our_ridge.coefficients):.4f}")
print(f"Coefficient Norm (sklearn): {np.linalg.norm(sklearn_ridge_pipeline.named_steps['ridge'].coef_):.4f}")

## Summary & Key Takeaways

### What We Learned:

1. **Mathematical Foundation**: Polynomial regression extends linear regression by transforming features into polynomial basis

2. **Implementation Details**: 
   - Feature transformation: x -> [1, x, x^2, ..., x^n]
   - Normal equation for closed-form solution
   - Numerical stability with Cholesky decomposition

3. **Degree Selection**:
   - Too low: underfitting (high bias)
   - Too high: overfitting (high variance)
   - Use cross-validation to find optimal degree

4. **Regularization**: Essential for higher degrees to prevent overfitting

### Key Insights:

- Polynomial regression is powerful for capturing non-linear relationships
- The bias-variance tradeoff is clearly visible with polynomial degree
- Our implementation matches sklearn closely
- Always validate with cross-validation before choosing final degree

### Best Practices:

1. Start with low degree, increase gradually
2. Use cross-validation for degree selection
3. Apply regularization for degree > 3
4. Scale features before polynomial expansion
5. Be cautious about extrapolation

### Next Steps:

- Try multivariate polynomial regression
- Explore splines for more flexible curves
- Implement gradient descent as alternative to normal equation
- Compare with other non-linear methods (KNN, Decision Trees)